# Pairwise Cross-Era Modeling

For each (train_dataset, test_dataset) pair we train one full pipeline using
the pair-specific feature set from `pairwise_feature_schema.json`. The
diagonal (train == test) uses an internal stratified 80/20 split so the
within-dataset score is held-out, not training-set fit. The off-diagonal
trains on the full training set and evaluates on the full test set.

Each saved pickle is self-contained: a fitted sklearn `Pipeline` (imputer
+ scaler if linear + classifier), the exact feature columns it was trained
on, and metadata so a future inference run can detect column drift.


In [1]:
import json
import sys
import time
from pathlib import Path

# Make the project root importable so `utils.*` resolves correctly.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from utils.modeling import (
    load_features_and_schema,
    run_pairwise_experiment,
    save_pairwise_models,
    results_to_matrix,
)

# Every figure produced anywhere in the project goes through this helper
# so it lands in report/figures/. Naming convention: "03_model__{descriptor}".
from utils.figures import save_fig

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR     = PROJECT_ROOT / "data" / "models"

DATASETS = ["cresci_2015", "cresci_2017", "twibot_2020", "twibot_2022"]

# Feature data (parquet preserves NaN-vs-0 + dtypes that CSV would drop)
# and the pairwise schema contract from notebook 02.
df, schema = load_features_and_schema(
    PROCESSED_DIR / "final_features" / "all_datasets_full_features.parquet",
    PROCESSED_DIR / "final_features" / "pairwise_feature_schema.json",
)

print(f"Feature frame: {len(df):,} users × {df.shape[1]} columns")
print(f"Pairwise schema: {len(schema)} (train, test) entries")
print(f"Datasets: {DATASETS}")


Feature frame: 1,031,495 users × 117 columns
Pairwise schema: 16 (train, test) entries
Datasets: ['cresci_2015', 'cresci_2017', 'twibot_2020', 'twibot_2022']


## Logistic Regression

In [2]:
MODEL_TYPE = "logreg"
out_dir    = MODEL_DIR / "logistic_regression"

start = time.time()
models, results = run_pairwise_experiment(df, schema, DATASETS, model_type=MODEL_TYPE)
print(f"\nTotal runtime: {time.time() - start:.2f}s\n")

auc_matrix = results_to_matrix(results, DATASETS, metric="auc")
acc_matrix = results_to_matrix(results, DATASETS, metric="accuracy")

print("AUC matrix (rows = train, cols = test):")
display(auc_matrix.round(4))

out_dir.mkdir(parents=True, exist_ok=True)
auc_matrix.to_csv(out_dir / "auc_matrix.csv")
acc_matrix.to_csv(out_dir / "acc_matrix.csv")

with open(out_dir / "results.json", "w") as f:
    json.dump(results, f, indent=2)

save_pairwise_models(models, out_dir, model_type=MODEL_TYPE)


/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  cresci_2015  → cresci_2015  (within-dataset 103 feat) AUC=0.9955  Acc=0.9868
  cresci_2015  → cresci_2017  (cross-era      32 feat) AUC=0.2663  Acc=0.4066
  cresci_2015  → twibot_2020  (cross-era      31 feat) AUC=0.7308  Acc=0.5628


/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  cresci_2015  → twibot_2022  (cross-era      33 feat) AUC=0.5398  Acc=0.6714
  cresci_2017  → cresci_2015  (cross-era      32 feat) AUC=0.6411  Acc=0.6870


/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


...

  twibot_2022  → twibot_2020  (cross-era      34 feat) AUC=0.7716  Acc=0.5978


/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  twibot_2022  → twibot_2022  (within-dataset 107 feat) AUC=0.8356  Acc=0.7645

Total runtime: 54.64s

AUC matrix (rows = train, cols = test):


,cresci_2015,cresci_2017,twibot_2020,twibot_2022
cresci_2015,0.9955,0.2663,0.7308,0.5398
cresci_2017,0.6411,0.9904,0.3091,0.4916
twibot_2020,0.5760,0.7446,0.8862,0.4897
twibot_2022,0.3281,0.8272,0.7716,0.8356


Saved 16 pickled bundles to /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/models/logistic_regression


## Ridge

In [3]:
MODEL_TYPE = "ridge"
out_dir    = MODEL_DIR / "ridge"

start = time.time()
models, results = run_pairwise_experiment(df, schema, DATASETS, model_type=MODEL_TYPE)
print(f"\nTotal runtime: {time.time() - start:.2f}s\n")

auc_matrix = results_to_matrix(results, DATASETS, metric="auc")
acc_matrix = results_to_matrix(results, DATASETS, metric="accuracy")

print("AUC matrix (rows = train, cols = test):")
display(auc_matrix.round(4))

out_dir.mkdir(parents=True, exist_ok=True)
auc_matrix.to_csv(out_dir / "auc_matrix.csv")
acc_matrix.to_csv(out_dir / "acc_matrix.csv")

with open(out_dir / "results.json", "w") as f:
    json.dump(results, f, indent=2)

save_pairwise_models(models, out_dir, model_type=MODEL_TYPE)


/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To

  cresci_2015  → cresci_2015  (within-dataset 103 feat) AUC=0.9955  Acc=0.9868
  cresci_2015  → cresci_2017  (cross-era      32 feat) AUC=0.2663  Acc=0.4066
  cresci_2015  → twibot_2020  (cross-era      31 feat) AUC=0.7308  Acc=0.5628


/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  cresci_2015  → twibot_2022  (cross-era      33 feat) AUC=0.5398  Acc=0.6714
  cresci_2017  → cresci_2015  (cross-era      32 feat) AUC=0.6411  Acc=0.6870


/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To

...

  twibot_2022  → twibot_2020  (cross-era      34 feat) AUC=0.7716  Acc=0.5978


/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  twibot_2022  → twibot_2022  (within-dataset 107 feat) AUC=0.8356  Acc=0.7645

Total runtime: 52.28s

AUC matrix (rows = train, cols = test):


,cresci_2015,cresci_2017,twibot_2020,twibot_2022
cresci_2015,0.9955,0.2663,0.7308,0.5398
cresci_2017,0.6411,0.9904,0.3091,0.4916
twibot_2020,0.5760,0.7446,0.8862,0.4897
twibot_2022,0.3281,0.8272,0.7716,0.8356


Saved 16 pickled bundles to /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/models/ridge


## Lasso

In [4]:
MODEL_TYPE = "lasso"
out_dir    = MODEL_DIR / "lasso"

start = time.time()
models, results = run_pairwise_experiment(df, schema, DATASETS, model_type=MODEL_TYPE)
print(f"\nTotal runtime: {time.time() - start:.2f}s\n")

auc_matrix = results_to_matrix(results, DATASETS, metric="auc")
acc_matrix = results_to_matrix(results, DATASETS, metric="accuracy")

print("AUC matrix (rows = train, cols = test):")
display(auc_matrix.round(4))

out_dir.mkdir(parents=True, exist_ok=True)
auc_matrix.to_csv(out_dir / "auc_matrix.csv")
acc_matrix.to_csv(out_dir / "acc_matrix.csv")

with open(out_dir / "results.json", "w") as f:
    json.dump(results, f, indent=2)

save_pairwise_models(models, out_dir, model_type=MODEL_TYPE)


/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


  cresci_2015  → cresci_2015  (within-dataset 103 feat) AUC=0.9983  Acc=0.9859


/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


  cresci_2015  → cresci_2017  (cross-era      32 feat) AUC=0.2539  Acc=0.4050


/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


...

  twibot_2022  → twibot_2020  (cross-era      34 feat) AUC=0.7721  Acc=0.5971


/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


  twibot_2022  → twibot_2022  (within-dataset 107 feat) AUC=0.8356  Acc=0.7645

Total runtime: 3657.35s

AUC matrix (rows = train, cols = test):


,cresci_2015,cresci_2017,twibot_2020,twibot_2022
cresci_2015,0.9983,0.2539,0.6778,0.5224
cresci_2017,0.6357,0.9906,0.3181,0.4834
twibot_2020,0.5777,0.7565,0.8863,0.4940
twibot_2022,0.3263,0.8270,0.7721,0.8356


Saved 16 pickled bundles to /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/models/lasso


## Random Forest

In [5]:
MODEL_TYPE = "rf"
out_dir    = MODEL_DIR / "random_forest"

start = time.time()
models, results = run_pairwise_experiment(df, schema, DATASETS, model_type=MODEL_TYPE)
print(f"\nTotal runtime: {time.time() - start:.2f}s\n")

auc_matrix = results_to_matrix(results, DATASETS, metric="auc")
acc_matrix = results_to_matrix(results, DATASETS, metric="accuracy")

print("AUC matrix (rows = train, cols = test):")
display(auc_matrix.round(4))

out_dir.mkdir(parents=True, exist_ok=True)
auc_matrix.to_csv(out_dir / "auc_matrix.csv")
acc_matrix.to_csv(out_dir / "acc_matrix.csv")

with open(out_dir / "results.json", "w") as f:
    json.dump(results, f, indent=2)

save_pairwise_models(models, out_dir, model_type=MODEL_TYPE)


  cresci_2015  → cresci_2015  (within-dataset 103 feat) AUC=0.9997  Acc=0.9887
  cresci_2015  → cresci_2017  (cross-era      32 feat) AUC=0.5470  Acc=0.7391
  cresci_2015  → twibot_2020  (cross-era      31 feat) AUC=0.5708  Acc=0.5593
  cresci_2015  → twibot_2022  (cross-era      33 feat) AUC=0.5658  Acc=0.5459
  cresci_2017  → cresci_2015  (cross-era      32 feat) AUC=0.5021  Acc=0.6636
  cresci_2017  → cresci_2017  (within-dataset 103 feat) AUC=0.9995  Acc=0.9934
  cresci_2017  → twibot_2020  (cross-era      32 feat) AUC=0.6302  Acc=0.5551
  cresci_2017  → twibot_2022  (cross-era      33 feat) AUC=0.6266  Acc=0.5970
  twibot_2020  → cresci_2015  (cross-era      31 feat) AUC=0.3973  Acc=0.7216
  twibot_2020  → cresci_2017  (cross-era      32 feat) AUC=0.4889  Acc=0.6455
  twibot_2020  → twibot_2020  (within-dataset 107 feat) AUC=0.8840  Acc=0.8187
  twibot_2020  → twibot_2022  (cross-era      34 feat) AUC=0.5298  Acc=0.2760
  twibot_2022  → cresci_2015  (cross-era      33 feat) AUC=0.

,cresci_2015,cresci_2017,twibot_2020,twibot_2022
cresci_2015,0.9997,0.5470,0.5708,0.5658
cresci_2017,0.5021,0.9995,0.6302,0.6266
twibot_2020,0.3973,0.4889,0.8840,0.5298
twibot_2022,0.3334,0.7593,0.7765,0.8516


Saved 16 pickled bundles to /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/models/random_forest


## XGBoost

In [6]:
MODEL_TYPE = "xgb"
out_dir    = MODEL_DIR / "xgboost"

start = time.time()
models, results = run_pairwise_experiment(df, schema, DATASETS, model_type=MODEL_TYPE)
print(f"\nTotal runtime: {time.time() - start:.2f}s\n")

auc_matrix = results_to_matrix(results, DATASETS, metric="auc")
acc_matrix = results_to_matrix(results, DATASETS, metric="accuracy")

print("AUC matrix (rows = train, cols = test):")
display(auc_matrix.round(4))

out_dir.mkdir(parents=True, exist_ok=True)
auc_matrix.to_csv(out_dir / "auc_matrix.csv")
acc_matrix.to_csv(out_dir / "acc_matrix.csv")

with open(out_dir / "results.json", "w") as f:
    json.dump(results, f, indent=2)

save_pairwise_models(models, out_dir, model_type=MODEL_TYPE)


  cresci_2015  → cresci_2015  (within-dataset 103 feat) AUC=0.9997  Acc=0.9906
  cresci_2015  → cresci_2017  (cross-era      32 feat) AUC=0.6581  Acc=0.5452
  cresci_2015  → twibot_2020  (cross-era      31 feat) AUC=0.6426  Acc=0.4943
  cresci_2015  → twibot_2022  (cross-era      33 feat) AUC=0.6659  Acc=0.8281
  cresci_2017  → cresci_2015  (cross-era      32 feat) AUC=0.5209  Acc=0.5782
  cresci_2017  → cresci_2017  (within-dataset 103 feat) AUC=0.9996  Acc=0.9941
  cresci_2017  → twibot_2020  (cross-era      32 feat) AUC=0.6194  Acc=0.5959
  cresci_2017  → twibot_2022  (cross-era      33 feat) AUC=0.6592  Acc=0.7071
  twibot_2020  → cresci_2015  (cross-era      31 feat) AUC=0.4815  Acc=0.6146
  twibot_2020  → cresci_2017  (cross-era      32 feat) AUC=0.5024  Acc=0.5125
  twibot_2020  → twibot_2020  (within-dataset 107 feat) AUC=0.8969  Acc=0.8331
  twibot_2020  → twibot_2022  (cross-era      34 feat) AUC=0.5420  Acc=0.3409
  twibot_2022  → cresci_2015  (cross-era      33 feat) AUC=0.

,cresci_2015,cresci_2017,twibot_2020,twibot_2022
cresci_2015,0.9997,0.6581,0.6426,0.6659
cresci_2017,0.5209,0.9996,0.6194,0.6592
twibot_2020,0.4815,0.5024,0.8969,0.5420
twibot_2022,0.2940,0.8009,0.7827,0.8582


Saved 16 pickled bundles to /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/models/xgboost


## Linear SVM

In [7]:
MODEL_TYPE = "svm"
out_dir    = MODEL_DIR / "linear_svm"

start = time.time()
models, results = run_pairwise_experiment(df, schema, DATASETS, model_type=MODEL_TYPE)
print(f"\nTotal runtime: {time.time() - start:.2f}s\n")

auc_matrix = results_to_matrix(results, DATASETS, metric="auc")
acc_matrix = results_to_matrix(results, DATASETS, metric="accuracy")

print("AUC matrix (rows = train, cols = test):")
display(auc_matrix.round(4))

out_dir.mkdir(parents=True, exist_ok=True)
auc_matrix.to_csv(out_dir / "auc_matrix.csv")
acc_matrix.to_csv(out_dir / "acc_matrix.csv")

with open(out_dir / "results.json", "w") as f:
    json.dump(results, f, indent=2)

save_pairwise_models(models, out_dir, model_type=MODEL_TYPE)


  cresci_2015  → cresci_2015  (within-dataset 103 feat) AUC=0.9933  Acc=0.9859
  cresci_2015  → cresci_2017  (cross-era      32 feat) AUC=0.2630  Acc=0.4110
  cresci_2015  → twibot_2020  (cross-era      31 feat) AUC=0.7527  Acc=0.5784
  cresci_2015  → twibot_2022  (cross-era      33 feat) AUC=0.5614  Acc=0.6708
  cresci_2017  → cresci_2015  (cross-era      32 feat) AUC=0.6350  Acc=0.6821
  cresci_2017  → cresci_2017  (within-dataset 103 feat) AUC=0.9906  Acc=0.9537
  cresci_2017  → twibot_2020  (cross-era      32 feat) AUC=0.3620  Acc=0.5525
  cresci_2017  → twibot_2022  (cross-era      33 feat) AUC=0.4915  Acc=0.8162
  twibot_2020  → cresci_2015  (cross-era      31 feat) AUC=0.5963  Acc=0.2737
  twibot_2020  → cresci_2017  (cross-era      32 feat) AUC=0.7222  Acc=0.3898
  twibot_2020  → twibot_2020  (within-dataset 107 feat) AUC=0.8859  Acc=0.8238
  twibot_2020  → twibot_2022  (cross-era      34 feat) AUC=0.4887  Acc=0.1660
  twibot_2022  → cresci_2015  (cross-era      33 feat) AUC=0.

,cresci_2015,cresci_2017,twibot_2020,twibot_2022
cresci_2015,0.9933,0.2630,0.7527,0.5614
cresci_2017,0.6350,0.9906,0.3620,0.4915
twibot_2020,0.5963,0.7222,0.8859,0.4887
twibot_2022,0.3322,0.8369,0.7727,0.8336


Saved 16 pickled bundles to /Users/ahteshamalvi/Files/CMSC/CMSC396H/Twibot_Detection_Research/data/models/linear_svm
